# Leaky modes of a step-index fiber

Step-index fibers typically have guided modes and *leaky modes*. Leaky modes are also called *resonances* or quasinormal modes. They are defined as a nontrivial function 
$\varphi$ that satisfies, together with some complex number $\beta$, the following Helmholtz equation on the entire plane containing a fiber's transverse cross section, 

$$\tag{1}
\Delta \varphi + k^2 n^2 \varphi = \beta^2 \varphi \qquad \text{ in all } \mathbb R^2.
$$

Instead of a boundary condition near the fiber boundary, leaky modes $\varphi$ satisfy the condition that they are **outgoing** at infinity.

The definition of an outgoing solution of $\Delta \varphi + (k^2 n^2 -\beta^2) \varphi = 0$ when $\kappa^2 = k^2 n^2 -\beta^2$ is real is simple and can be found in textbooks. One equivalent characterization of outgoing $\varphi$ when $\kappa$ is real is that $\varphi$ satisfies the [Sommerfeld radiation condition](https://en.wikipedia.org/wiki/Sommerfeld_radiation_condition). For complex $\kappa$ however, the definition of outgoing is more delicate. A mathematically correct approach is to use complex analysis to define an outgoing $\varphi$ as a meromorphic continuation of an outgoing solution obtained in the real $\kappa$ case. Leaky modes have complex $\beta$. In our numerical methods to compute leaky modes, the perfectly matched layer (PML) selects outgoing solutions by damping them exponentially. In our semi-analytical methods, we select Hankel functions that are outgoing to match the fiber core solutions.

The modal solutions of (1) yield Helmholtz solutions $u(x, y,z) = \varphi(x, y) e^{i \beta z}$ propagating along the longitudinal ($z$) direction of the fiber. Confinement loss (CL), usually expressed in decibels (dB) per meter, is defined as 

$$
CL = -10 \log_{10} \frac{P(1)}{P(0)}
$$

where $P(z)$ is the power in the fiber cross section at $z$. For a leaky mode, viewing the power as proportional to $|e^{i\beta z}|^2$,  its CL can be estimated from the imaginary part of the propagation constant $\text{Im}(\beta)$ by 

$$
CL = 20 \frac{\text{Im}(\beta)}{\ln(10)}.
$$

For this reason, computing the imaginary part of $\beta$ for leaky modes is important for correctly estimating confinement losses. 

```{index} loss; confinement loss formula
```

```{index} loss; db/m from propagation constant
```

In this notebook we present semi-analytical facilities based on exact leaky mode expressions available for step-index fibers, as well as fully numerical facilities using finite elements to compute the same modes approximately. 

## Semi-analytical leaky mode finder

Leaky modes are computed in the non-dimensional $Z$-plane, while the guided modes are computed in the non-dimensional $X$-plane. Recall from [Notebook 1.1](1_1_stepindex_exact.ipynb) that non-dimensional $X$-values are related to dimensional $\beta$-values by $X = r_{\text{core}}\sqrt{n_{\text{core}}^2 k^2-\beta^2}.$ The new non-dimensional variable $Z$ is related to $X$ via the fiber V-number:

```{index} X and propagation constant
``` 

$$\tag{2}
X^2 - Z^2 = V^2.
$$

```{index} Z and X
``` 


The facility to compute $Z$ values for leaky modes, which then give the leaky $\beta$ values, is shown next:

In [ ]:
from fibermode import StepIndexExact
import numpy as np
import logging,  warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
f = StepIndexExact('Nufern_Yb')
z = f.leaky_propagation_constants(0)
Z = np.array(z)
Z

Using equation (2), we can compute $\beta$ values from these $Z$ values. The `StepIndexExact` class has a method to do just that. Here are the corresponding dimensional $\beta$ values:

In [ ]:
f.ZtoBeta(Z)  # convert to dimensional propagation constants

For each value of $Z$ (or $\beta$), the accompanying mode function can be visualized as shown next.

In [ ]:
X, Y, F, mode,  = f.visualize_leaky_mode(z[0], 0)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm

fig, ax = plt.subplots(1, 2, subplot_kw={'projection': '3d'})
fig.set_size_inches(10, 7)
fig.tight_layout()

ax[0].plot_surface(X, Y, F.real, cmap=cm.coolwarm, linewidth=0);
ax[0].view_init(azim=-60, elev=68)

ax[1].plot_surface(X, Y, F.imag, cmap=cm.coolwarm);
ax[1].view_init(azim=-60, elev=68)

## Numerical leaky mode finder

Using finite elements and the perfectly matched layer (PML), the `StepIndex` class of `fibermode` provides numerical routines to compute the same leaky modes. PML allows us to truncate the infinite domain $\mathbb R^2$ to a bounded circular computational domain. It is important to note that *the numerical modes are only accurate in the region without PML* since PML deforms and attenuates the mode within it to eliminate potential reflections from the artificial boundary.

In [ ]:
from fibermode import StepIndex
from ngsolve.webgui import Draw

Using the same class `StepIndex` that we used for numerical guided mode computation in [Notebook 1.2](1_2_stepindex.ipynb), we can now compute leaky modes by specifying PML parameters.

In [ ]:
fiber = StepIndex(fibername='Nufern_Yb', 
                  R=2,     
                  Rout=8,   # PML is set in R < r < Rout
                  refine=1)

Next, we call the `leakymode` method, an algorithm applicable to many fibers. It uses a *frequency-dependent perfectly matched layer (PML) and a polynomial eigensolver* described in [[1]](#references), which builds on the frequency-dependent PML approach of [[2]](#references). The PML strength and an approximate search region for finding leaky modes in the $Z$-plane must be given as inputs. 

```{index} PML; frequency-dependent
```

```{index} eigensolver; polynomial
```

In [ ]:
zh, y, yl, beta, _, _ = fiber.leakymode(p=3,          # degree of finite  elements
                                        rad=0.3,      # search radius in Z-plane
                                        ctr=5.4-1.3j, # center of search circle in Z-plane
                                        alpha=5,      # PML strength parameter
                                        verbose=False)

We see that the computed $Z$-value is close to one of the values found in the previous section through semi-analytical computation. The accompanying numerical leaky modes can be visualized using ngsolve as follows.

In [ ]:
yh = y.gridfun()
yh_real = -10 * yh.real
Draw(yh_real, fiber.mesh, deformation=True, euler_angles=[-55, 10,10]);

This plot shows that in the non-PML region $r < R$, the numerically found mode indeed approximates the semi-analytical mode computed earlier by different methods. (In the PML region $R < r < R_{\text{out}}$, the mode decays rapidly.)

## Spectral locations in the complex plane

For the fiber we've been considering, let us examine what the computations show us about the structure of the spectrum in the complex plane. We can visualize the spectrum either in the $\beta$-plane or in the non-dimensional $Z$-plane. We begin with the $Z$-plane. We will be able to discern the spectral structure when we plot all the computed $Z$ values together. 

In [ ]:
# compute many leaky modes 

Z = [np.array(z)]
for i in range(1, 11):
  Z += [np.array(f.leaky_propagation_constants(i))]

In [ ]:
print('Leaky mode Z values:')
[print('l=%2d:'%i, z) for i, z in enumerate(Z)];

In [ ]:
# compute all guided modes

X = []
for i in range(3): 
    X += f.propagation_constants(i, loglevel=logging.ERROR)

In [ ]:
Zguided2 = np.array(X)**2 - f.fiberV()**2
Zguided = 1j  * np.sqrt(-Zguided2)
Zguided

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(8, 6)
ax.set_title('Spectrum of the optical fiber in the $Z$ plane')
ax.set_xlabel('real axis')
ax.set_ylabel('imaginary axis')

# draw real and imaginary axis of the complex plane
ax.plot([0, 0], [-2, 5], 'c');
ax.plot([-1, 13], [0, 0], 'c');

# plot the guided mode eigenvalues
ax.plot(Zguided.real, Zguided.imag, 'k*', label='Guided')

# plot the leakymode eigenvalues
for l in range(10):
    lab = 'Leaky(%d)' % l
    ax.scatter(np.array(Z[l]).real, np.array(Z[l]).imag, label=lab)
ax.grid(True); ax.legend();

The physical propagation constants $\beta$ are large and are located in a different region in the complex plane. Let us visualize the same spectrum in the $\beta$-plane.

In [ ]:
betas = []
for z in Z:
    betas.append(f.ZtoBeta(np.array(z)))

In [ ]:
print('Leaky mode beta values:')
[print('l=%2d:'%i, beta) for i, beta in enumerate(betas)];

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(8, 6)
ax.set_title('Spectrum of the optical fiber in the $\\beta$ plane')
ax.set_xlabel('real part')
ax.set_ylabel('imaginary part')
ax.plot(f.XtoBeta(X), [0]*len(X), 'k*', label='Guided')
for l in range(10):
    lab = 'Leaky(%d)' % l
    ax.scatter(betas[l].real, betas[l].imag, label=lab)
ax.grid(True); ax.legend();

These plots show the typical structure of the spectrum of a step-index optical fiber. To reinforce this point, we conclude by showing schematic diagrams of the spectrum in both the $\beta$-plane and the $Z$-plane.

Typical diagrams of propagation constant locations ($\beta$-values) in the complex plane take the following form, where we have separated guided, leaky, evanescent, and radiation parts of the spectrum by different colors:

```{index} modes; evanescent
```
```{index} modes; radiation
```
```{index} modes; leaky
```

<img src='./figs/spectrumBeta.png' width="250" align="center"/>

In contrast, the non-dimensional $Z$ value locations in the complex plane have a different structure. Since the optical $Z^2$ value can be equivalently viewed as a Schrödinger spectral point, and since the essential spectrum of the Laplacian is unperturbed by a bounded potential well, we first mark the essential spectrum in the plot (in red). The guided modes are now on the imaginary axis in the $Z$-plane, while the leaky modes hover just below the real axis.

<img src='./figs/spectrumZ.png' width="250" align="center"/>

<a id='references'></a>
## References

[1] J. Gopalakrishnan, C. Parker, and P. Vandenberge, "Computing leaky modes of optical fibers using a FEAST algorithm for polynomial eigenproblems," *Wave Motion*, vol. 101, p. 102826, 2021. DOI: [10.1016/j.wavemoti.2021.102826](https://doi.org/10.1016/j.wavemoti.2021.102826)

[2] L. Nannen and M. Wess, "Computing scattering resonances using perfectly matched layers with frequency dependent scaling functions," *BIT Numer. Math.*, vol. 58, pp. 373–395, 2018. DOI: [10.1007/s10543-017-0694-1](https://doi.org/10.1007/s10543-017-0694-1)